# 1 Imports

In [1]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

from funciones import *
import joblib

c:\Users\Jorge\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2 Carga de los datos

In [2]:
SPOOF = "..\\..\\Dataset\\Features\\spoof_features.csv"
BONAFIDE = '..\\..\\Dataset\\Features\\bonafide_features.csv'

RANDOM_STATE = 12
TEST_SPLIT = 0.2

In [3]:
df = carga_datos(ruta_bon=BONAFIDE, ruta_spo=SPOOF, random_state=RANDOM_STATE)

In [4]:
X = df.drop(columns=['label', 'filename'])
y = df['label']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SPLIT, random_state=RANDOM_STATE, stratify=y)

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3 Entrenamiento de modelos

## 3.1 Kernel lineal

In [7]:
# SVM con kernel lineal (modelo simple, pocos parámetros)
svm_linear = SVC(
    kernel='linear',           # Frontera de decisión lineal
    C=1.0,                     # Parámetro de regularización
    probability=True,           # Para obtener probabilidades (necesario para AUC-ROC)
    random_state=RANDOM_STATE,
    class_weight='balanced'    # Compensación de clases desbalanceadas
)

svm_linear.fit(X_train_scaled, y_train)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [8]:
# Predicciones
y_pred_linear = svm_linear.predict(X_test_scaled)
y_pred_proba_linear = svm_linear.predict_proba(X_test_scaled)[:, 1]

In [9]:
# Evaluación
linear_metrics = evaluacion_modelo(
    "SVM - Lineal", 
    y_test, 
    y_pred_linear, 
    y_pred_proba_linear,
)


 EVALUACIÓN: SVM - Lineal

MÉTRICAS PRINCIPALES:
   • Accuracy:  1.0000  (Porcentaje total de aciertos)
   • Precision: 1.0000  (De los que dije que eran reales, ¿cuántos lo eran?)
   • Recall:    1.0000  (De los reales, ¿cuántos detecté?)
   • F1-Score:  1.0000  (Balance precision-recall)
   • AUC-ROC:   1.0000  (Capacidad discriminativa)
   • MCC:       1.0000  (Coeficiente de correlación de Matthews)

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    Real (1)     1.0000    1.0000    1.0000        10
      IA (0)     1.0000    1.0000    1.0000        10

    accuracy                         1.0000        20
   macro avg     1.0000    1.0000    1.0000        20
weighted avg     1.0000    1.0000    1.0000        20


ANÁLISIS DE ERRORES:
   • Verdaderos Negativos (Reales bien clasificados): 10
   • Verdaderos Positivos (IA bien clasificados): 10
   • Falsos Positivos (Reales clasificados como IA): 0
   • Falsos Negativos (IA clasificados como Reales): 0

### 3.2 Kernel no lineal

In [10]:
# SVM con kernel RBF (Radial Basis Function) - Captura no linealidades
svm_rbf = SVC(
    kernel='rbf',              # Kernel no lineal
    C=1.0,                     # Regularización
    gamma='scale',              # Coeficiente del kernel ('scale' = 1/(n_features * X.var()))
    probability=True,
    random_state=RANDOM_STATE,
    class_weight='balanced'
)

svm_rbf.fit(X_train_scaled, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [11]:
# Predicciones
y_pred_rbf = svm_rbf.predict(X_test_scaled)
y_pred_proba_rbf = svm_rbf.predict_proba(X_test_scaled)[:, 1]

In [12]:
# Evaluación
rbf_metrics = evaluacion_modelo(
    "SVM - RBF", 
    y_test, 
    y_pred_rbf, 
    y_pred_proba_rbf,
    save_figures=False
)


 EVALUACIÓN: SVM - RBF

MÉTRICAS PRINCIPALES:
   • Accuracy:  1.0000  (Porcentaje total de aciertos)
   • Precision: 1.0000  (De los que dije que eran reales, ¿cuántos lo eran?)
   • Recall:    1.0000  (De los reales, ¿cuántos detecté?)
   • F1-Score:  1.0000  (Balance precision-recall)
   • AUC-ROC:   1.0000  (Capacidad discriminativa)
   • MCC:       1.0000  (Coeficiente de correlación de Matthews)

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    Real (1)     1.0000    1.0000    1.0000        10
      IA (0)     1.0000    1.0000    1.0000        10

    accuracy                         1.0000        20
   macro avg     1.0000    1.0000    1.0000        20
weighted avg     1.0000    1.0000    1.0000        20


ANÁLISIS DE ERRORES:
   • Verdaderos Negativos (Reales bien clasificados): 10
   • Verdaderos Positivos (IA bien clasificados): 10
   • Falsos Positivos (Reales clasificados como IA): 0
   • Falsos Negativos (IA clasificados como Reales): 0
  

### 3.3 Cross validation

In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Creamos un pipeline con escalado y SVM para validación cruzada


svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Escalado dentro del pipeline
    ('svm', SVC(kernel='rbf', probability=True, 
                random_state=RANDOM_STATE, class_weight='balanced'))
])

In [14]:
# Validación cruzada con el pipeline
cv_scores_accuracy = cross_val_score(svm_pipeline, X_train, y_train, 
                                     cv=cv, scoring='accuracy')
cv_scores_f1 = cross_val_score(svm_pipeline, X_train, y_train, 
                               cv=cv, scoring='f1')
cv_scores_auc = cross_val_score(svm_pipeline, X_train, y_train, 
                                cv=cv, scoring='roc_auc')

In [15]:
print("\nResultados Validación Cruzada SVM RBF (5 folds):")
print(f"   • Accuracy: {cv_scores_accuracy.mean():.4f} (+/- {cv_scores_accuracy.std()*2:.4f})")
print(f"   • F1-Score: {cv_scores_f1.mean():.4f} (+/- {cv_scores_f1.std()*2:.4f})")
print(f"   • AUC-ROC:  {cv_scores_auc.mean():.4f} (+/- {cv_scores_auc.std()*2:.4f})")


Resultados Validación Cruzada SVM RBF (5 folds):
   • Accuracy: 1.0000 (+/- 0.0000)
   • F1-Score: 1.0000 (+/- 0.0000)
   • AUC-ROC:  1.0000 (+/- 0.0000)


### 3.4 Grid Search

In [16]:
param_grid = {
    'svm__C': [0.1, 1, 10, 100],           # Regularización (logarítmico)
    'svm__gamma': [0.001, 0.01, 0.1, 1, 'scale', 'auto'],  # Influencia del kernel
    'svm__kernel': ['rbf']                   # Nos centramos en RBF
}

In [17]:
# Pipeline con escalado incluido
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True, random_state=RANDOM_STATE, 
                class_weight='balanced'))
])

In [18]:
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


,estimator,Pipeline(step...m_state=12))])
,param_grid,"{'svm__C': [0.1, 1, ...], 'svm__gamma': [0.001, 0.01, ...], 'svm__kernel': ['rbf']}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [19]:
# Modelo optimizado
svm_optimized = grid_search.best_estimator_

In [24]:
for param in grid_search.param_grid.keys():
    print(f"{param}: {grid_search.best_params_[param]}")

svm__C: 1
svm__gamma: 0.01
svm__kernel: rbf


In [20]:
# Predicciones con modelo optimizado
y_pred_proba_optimized = svm_optimized.predict_proba(X_test)[:, 1]
y_pred_optimized = svm_optimized.predict(X_test)

In [21]:
# Evaluación
optimized_metrics = evaluacion_modelo(
    "SVM - Optimizado (RBF)", 
    y_test, 
    y_pred_optimized, 
    y_pred_proba_optimized,
)


 EVALUACIÓN: SVM - Optimizado (RBF)

MÉTRICAS PRINCIPALES:
   • Accuracy:  1.0000  (Porcentaje total de aciertos)
   • Precision: 1.0000  (De los que dije que eran reales, ¿cuántos lo eran?)
   • Recall:    1.0000  (De los reales, ¿cuántos detecté?)
   • F1-Score:  1.0000  (Balance precision-recall)
   • AUC-ROC:   1.0000  (Capacidad discriminativa)
   • MCC:       1.0000  (Coeficiente de correlación de Matthews)

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    Real (1)     1.0000    1.0000    1.0000        10
      IA (0)     1.0000    1.0000    1.0000        10

    accuracy                         1.0000        20
   macro avg     1.0000    1.0000    1.0000        20
weighted avg     1.0000    1.0000    1.0000        20


ANÁLISIS DE ERRORES:
   • Verdaderos Negativos (Reales bien clasificados): 10
   • Verdaderos Positivos (IA bien clasificados): 10
   • Falsos Positivos (Reales clasificados como IA): 0
   • Falsos Negativos (IA clasificados como 

In [22]:
# Obtenemos el modelo del pipeline
svm_model = svm_optimized.named_steps['svm']
n_support_vectors = svm_model.n_support_
support_vectors_count = sum(n_support_vectors)

print(f"Total de vectores de soporte: {support_vectors_count}")
print(f"Porcentaje del dataset: {support_vectors_count/len(X_train)*100:.2f}%")
print(f"Vectores clase Real (0): {n_support_vectors[0]}")
print(f"Vectores clase IA (1): {n_support_vectors[1]}")

Total de vectores de soporte: 45
Porcentaje del dataset: 56.25%
Vectores clase Real (0): 23
Vectores clase IA (1): 22


# 4 Guardado del mejor modelo

In [23]:
# Guardar el modelo
joblib.dump(svm_optimized, 'modelo_SVM.pkl')

['modelo_SVM.pkl']